# EXERCISE 2 - SENTIMENT ANALYSIS CLASSIFICATION TASK WITH Pytorch BERT

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gzip
import tarfile
from sklearn.datasets import load_files

In [2]:
# Download Data
!wget https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar xvzf aclImdb_v1.tar.gz > /dev/null

--2024-03-12 16:45:28--  https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
Resolving ai.stanford.edu (ai.stanford.edu)... 171.64.68.10
Connecting to ai.stanford.edu (ai.stanford.edu)|171.64.68.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 84125825 (80M) [application/x-gzip]
Saving to: ‘aclImdb_v1.tar.gz’

aclImdb_v1.tar.gz   100%[===================>]  80.23M  1.62MB/s    in 47s     

2024-03-12 16:46:16 (1.70 MB/s) - ‘aclImdb_v1.tar.gz’ saved [84125825/84125825]



In [3]:
# # Load files in google collab


# from sklearn.datasets import load_files

# movies_train = load_files(container_path="aclImdb_v1/aclImdb/train", encoding="utf-8")
# movies_test = load_files(container_path="aclImdb_v1/aclImdb/test", encoding="utf-8")


In [4]:
# from sklearn.datasets import load_files

# movies_train = load_files(container_path="/content/aclImdb/train", encoding="utf-8")
# movies_test = load_files(container_path="/content/aclImdb/test", encoding="utf-8")

In [2]:
from sklearn.datasets import load_files
movies_train = load_files(container_path="//home/aclImdb/train", encoding="utf-8")
movies_test = load_files(container_path="//home/aclImdb/test", encoding="utf-8")
# print(type(movies_train))

In [3]:
# Transform train dataset into a dataframe
data_train = {'reviews': movies_train.data, 'sentiment': movies_train.target}
df_train = pd.DataFrame(data_train)

# Transform test dataset into a dataframe
data_test = {'reviews': movies_test.data, 'sentiment': movies_test.target}
df_test = pd.DataFrame(data_test)

del movies_train, movies_test # free memory

print(df_train.iloc[:10])
print('----------------------------------------------------------------------------------------')
print(df_test.iloc[:10])

                                             reviews  sentiment
0  Full of (then) unknown actors TSF is a great b...          2
1  Amount of disappointment I am getting these da...          2
2  The future, we are told, are what we make of i...          2
3  Dan Katzir has produced a wonderful film that ...          1
4  If you want Scream or anything like the big-st...          1
5  Although its mold of 1949 appears somewhat mel...          2
6  Gloomy Sunday - Ein Lied von Liebe und Tod dir...          2
7  This movie was ridiculous. The plot is complet...          2
8  Why was this movie made? No doubt to sucker in...          2
9  Outlandish premise that rates low on plausibil...          0
----------------------------------------------------------------------------------------
                                             reviews  sentiment
0  Don't hate Heather Graham because she's beauti...          1
1  I don't know how this movie has received so ma...          0
2  I caught thi

Remove label 2

In [4]:
# Drop rows where 'sentiment' is 2 from train & test sets
df_train = df_train[df_train['sentiment'] != 2]
df_test = df_test[df_test['sentiment'] != 2]


df_test = df_test.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)

# Define the train dataset with 5000 rows by test set
df_train = pd.concat([df_train, df_test[:5000]])

print(f'The size of training set is: {len(df_train)}')
print(f'The size of test set is: {len(df_test[5000:])}')

The size of training set is: 30000
The size of test set is: 20000


In [5]:
from sklearn.utils import shuffle

# Shuffle dataframes because they are sorted.
df_train = shuffle(df_train)
df_train.reset_index(inplace=True, drop=True)
df_test = shuffle(df_test)
df_test.reset_index(inplace=True, drop=True)

df_train.head()

,reviews,sentiment
0,San Francisco is a big city with great acting ...,0
1,How could I best express my feelings about thi...,0
2,Simply not the quality I expected from Morris ...,0
3,"Ride with the Devil, like Ang Lee's later Brok...",1
4,"Bad actors, terrible script, totally unbelieva...",0


In [6]:
X_train = df_train.reviews
y_train = df_train.sentiment

In [7]:
# Split dev and test data
from sklearn.model_selection import train_test_split

X_dev, X_test, y_dev, y_test = train_test_split(df_test.reviews[5000:], df_test.sentiment[5000:], test_size=0.5, random_state=42)
print(f"Number of observations in dev set: {X_dev.shape[0]}")
print(f"Number of observations in test set: {X_test.shape[0]}")

Number of observations in dev set: 10000
Number of observations in test set: 10000


# Preprocessing

* Replace non-word characters and numbers with empty string.
* Substitute multiple spaces with single space.
* Convert to lowercase.
* Remove stop words.
* Stemming with WordNetLemmatizer.

In [8]:
# Imports we need for preprocessing
import re
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer

stemmer = WordNetLemmatizer()

# Import stop words for preprocessing
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /home/meizeus/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/meizeus/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/meizeus/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
def preprocess(contents):
  """
    Preprocesses a list of texts by removing non-word characters, single characters, extra spaces,
    converting to lowercase, splitting into words, lemmatizing, and reconstructing the documents.

    Parameters:
    - contents (list): A list of texts to be preprocessed.

    Returns:
    - pd.Series: A Pandas Series containing preprocessed documents.
    """

  docs = []

  for doc in contents:

      # Remove non-word (special) characters such as punctuation, numbers etc
      document = re.sub(r'\W', ' ', str(doc))

      # Remove HTML <BR>
      document = re.sub(r'\s+br\s+',' ', str(document))

      # Remove all single characters
      document = re.sub(r'\s+[a-zA-Z]\s+', ' ', document)

      # Remove numbers
      document = re.sub(r'\b\d+\b', ' ', document)

      # Substitute multiple spaces with single space
      document = re.sub(r'\s+', ' ', document, flags=re.I) #re.I -> ignore case

      # Convert to Lowercase
      document = document.lower()

      # Split the document based on whitespaces (--> List of words)
      word_list = word_tokenize(document)

      # word_list = document.split()

      word_list = [word for word in word_list if word not in (stop_words)]

      # Lemmatization
      # word_list = [stemmer.lemmatize(word) for word in word_list]

      # # Reconstruct the document by joining the words on each whitespace
      document = ' '.join(word_list)

      # Append all documents into a list 'docs'
      docs.append(document)
  return pd.Series(docs)

In [10]:
# Preprocess texts
X_train = preprocess(X_train)
X_dev = preprocess(X_dev)
X_test = preprocess(X_test)

In [11]:
# Example of a preprocessed text
print(f'The non processed text:\n {df_train.reviews.iloc[10]}')
print('------------------------------------------------------------------------------------')
print('VS')
print('------------------------------------------------------------------------------------')
print(f'The processed text:\n {X_train[10]}')

The non processed text:
 This movie was a major disappointment on direction, intellectual niveau, plot and in the way it dealt with its subject, painting. It is a slow moving film set like an episode of Wonder Years, with appalling lack of depth though. It also fails to deliver its message in a convincing manner.<br /><br />The approach to the subject of painting is very elite, limited to vague and subjective terms as "beauty". According to the makers of this movie, 'beauty' can be only experienced in Bob-Ross-style kitschy landscape paintings. Good art according to this film can be achieved by applying basic (like, primary school level) color theory and lots of sentiment. In parts the movie is offending, e.g. at a point it is stated (rather, celebrated by dancing on tables) that mentally handicapped people are not capable of having emotions or expressing them through painting, their works by definition being worthless 'bullshit' (quote).<br /><br />I do not understand how the movie co

In [15]:
print(X_train.shape,X_dev.shape,X_test.shape)

(30000,) (10000,) (10000,)


## Baseline Model (Logistic Regression with  TF-IDF and SVD)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Use unigram & bi-gram tf*idf features
vectorizer = TfidfVectorizer(ngram_range=(1, 3),
                             max_features = 5000, sublinear_tf=True)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_dev_tfidf = vectorizer.transform(X_dev)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape, type(X_train_tfidf))

(30000, 5000) <class 'scipy.sparse._csr.csr_matrix'>


In [ ]:
# Reduce dimensionality using SVD 5000 --> 1000
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=1000, random_state=4321)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_dev_svd = svd.transform(X_dev_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

print(X_train_svd.shape, type(X_train_svd))

(30000, 1000) <class 'numpy.ndarray'>


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_curve, auc
clf = LogisticRegression()
clf.fit(X_train_svd.tolist(), y_train)

# Validation set prediction
predictions = clf.predict(X_dev_svd.tolist())
print(f"\t\tClassification Report for developement set:\n\n {classification_report(y_dev, predictions)}")

# Test set prediction
predictions2 = clf.predict(X_test_svd.tolist())
print(f"\n\n\t\tClassification Report for testing set:\n\n {classification_report(y_test, predictions2)}")

		Classification Report for developement set:

               precision    recall  f1-score   support

           0       0.89      0.87      0.88      4955
           1       0.88      0.90      0.89      5045

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



		Classification Report for testing set:

               precision    recall  f1-score   support

           0       0.90      0.89      0.89      5008
           1       0.89      0.90      0.90      4992

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



### TOKENIZATION

In [16]:
def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

In [18]:
# !pip install spacy
!python -m spacy download en_core_web_lg

import spacy
import string
import numpy as np
from tqdm import tqdm

nlp = spacy.load('en_core_web_lg', disable=["tagger", "parser", "ner", "lemmatizer"])
from spacy.lang.en.stop_words import STOP_WORDS
nlp.add_pipe('sentencizer')


# Create a function that uses spacy to tokenize sentences and returns a nested list of tokenized sentences
def tokenize_data (data):
  """
    Tokenize our data
    :param data: list of documents
    :return: nested list of the tokenized documents
    """
  data_tokenized = []
  for idx in tqdm(range(len(data))):
    doc = nlp(data[idx])
    tokens = []
    for sent in doc.sents:
      for tok in sent:
        if '\n' in tok.text or "\t" in tok.text or "--" in tok.text or "*" in tok.text or\
        tok.text.lower() in STOP_WORDS or tok.text in string.punctuation or\
          all(x in string.punctuation for x in tok.text) or is_number(tok.text):
          continue
        if tok.text.strip():
          tokens.append(tok.text.replace('"',"'").strip().lower())
    data_tokenized.append(tokens)
  return data_tokenized

X_train_tokenized = tokenize_data(X_train)
X_dev_tokenized = tokenize_data(X_dev)
X_test_tokenized = tokenize_data(X_test)



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 1.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


 14%|█▍        | 1388/10000 [00:15<01:36, 89.64it/s]


KeyboardInterrupt: 

In [ ]:
X_train_clean= [" ".join(x) for x in X_train_tokenized]
X_dev_clean = [" ".join(x) for x in X_dev_tokenized]
X_test_clean = [" ".join(x) for x in X_test_tokenized]

In [ ]:
from datasets.dataset_dict import DatasetDict
from datasets import Dataset

schema = {'train':Dataset.from_dict({'label':y_train,'text':X_train_clean}),
     'val':Dataset.from_dict({'label':y_dev,'text':X_dev_clean}),
     'test':Dataset.from_dict({'label':y_test,'text':X_test_clean}),
     }

# Create the sumed up dataframe
movies_dataset = DatasetDict(schema)

In [ ]:
movies_dataset

In [ ]:
movies_dataset["train"][133]

GET MAX SEQUENCE

In [ ]:
mean_words = np.mean([len(x) for x in X_train_tokenized])
std_of_words = np.std([len(x) for x in X_train_tokenized])

# Get mean and std of sequence length on trainning set
print(f"Mean: {np.mean([len(x) for x in X_train_tokenized]): .2f}")
print(f"Std: {np.std([len(x) for x in X_train_tokenized]): .2f}")

In [ ]:
y_dev.shape,y_test.shape

* LETS KEEP THE REVIEWS WITH MAX LENGTH= mean_words + (2 * std_of_word_counts)

In [ ]:
print(f"{[len(x) for x in X_train_tokenized]}")

# Define a function that drops the rows not meeting the criteria
def estimate_max_seq_len(data_tokenized, targets, mean_w=mean_words, std_w=std_of_words):
    # Create a DataFrame to work with
    data = pd.DataFrame(columns=['data_tokenized', 'word_counts', 'targets'])

    # Assign the tokenized text to a column inside the dataframe
    data["data_tokenized"] =  data_tokenized
    # Assign the word counts for each row in a second column
    data["word_counts"] = [len(x) for x in data_tokenized]

    # Assign target variables
    data["targets"] = targets.tolist()

    display(data.head())

    # Custom formula of estimating the MAX_words_Length
    MAX_words_length = mean_w + 2*std_w

    # Drop the rows not meeting the criteria
    data = data[data.word_counts <= MAX_words_length]

    return [' '.join(x) for x in data.data_tokenized], data.targets

X_train, y_train = estimate_max_seq_len(X_train_tokenized, y_train)
X_dev, y_dev = estimate_max_seq_len(X_dev_tokenized, y_dev)
X_test, y_test = estimate_max_seq_len(X_test_tokenized, y_test)



### CHECK COMPATIBILITY

In [1]:
import torch
print(torch.cuda.is_available())

True


In [12]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print("Cuda is Availabe")
else:
    print("Cuda Can't be found")

True
Cuda is Availabe


In [13]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

2024-03-12 21:46:11.158515: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-12 21:46:11.303341: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-03-12 21:46:11.821935: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: :/home/meizeus/miniconda3/envs/tf_env/lib/
2024-03-12 21:46:11.823501: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_

In [14]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, return_tensors='pt', max_length=self.max_length, padding='max_length', truncation=True)
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'label': torch.tensor(label)}

In [15]:
class BERTClassifier(nn.Module):
    def __init__(self, bert_model_name, num_classes):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        logits = self.fc(x)
        return logits

In [16]:
def train(model, data_loader, optimizer, scheduler, device):
    model.train()
    for batch in data_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

In [17]:
def evaluate(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
    return accuracy_score(actual_labels, predictions), classification_report(actual_labels, predictions)

In [18]:
def predict_sentiment(text, model, tokenizer, device, max_length=266):
    model.eval()
    encoding = tokenizer(text, return_tensors='pt', max_length=max_length, padding='max_length', truncation=True)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
    return "positive" if preds.item() == 1 else "negative"

In [19]:
# Set up parameters
bert_model_name = 'distilbert-base-uncased'
num_classes = 2
max_length = 266
batch_size = 16
num_epochs = 7
learning_rate = 2e-5

In [20]:
tokenizer = BertTokenizer.from_pretrained(bert_model_name)
tokenizer = BertTokenizer.from_pretrained(bert_model_name)
train_dataset = TextClassificationDataset(X_train,y_train, tokenizer, max_length)
val_dataset = TextClassificationDataset(X_dev, y_dev, tokenizer, max_length)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DistilBertTokenizer'. 
The class this function is called from is 'BertTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DistilBertTokenizer'. 
The class this function is called from is 'BertTokenizer'.


In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTClassifier(bert_model_name, num_classes).to(device)

You are using a model of type distilbert to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing BertModel: ['distilbert.transformer.layer.4.attention.v_lin.weight', 'vocab_transform.bias', 'vocab_layer_norm.bias', 'distilbert.transformer.layer.5.attention.v_lin.weight', 'distilbert.transformer.layer.5.ffn.lin1.bias', 'distilbert.transformer.layer.5.ffn.lin2.weight', 'distilbert.transformer.layer.2.ffn.lin2.bias', 'distilbert.transformer.layer.3.attention.out_lin.weight', 'distilbert.transformer.layer.1.output_layer_norm.bias', 'distilbert.transformer.layer.3.ffn.lin2.weight', 'distilbert.transformer.layer.3.attention.q_lin.weight', 'distilbert.transformer.layer.5.attention.q_lin.bias', 'distilbert.transformer.layer.0.output_layer_norm.bias', 'distilbert.transformer.layer.1.ffn.lin2.weight', 'vocab_layer_norm.weight', 'distilbert.tran

In [22]:
optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_dataloader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

/home/meizeus/miniconda3/envs/tf_env/lib/python3.10/site-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [23]:
for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        train(model, train_dataloader, optimizer, scheduler, device)
        accuracy, report = evaluate(model, val_dataloader, device)
        print(f"Validation Accuracy: {accuracy:.4f}")
        print(report)

Epoch 1/7


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "bert_classifier.pth")s

In [ ]:
# Test sentiment prediction
test_text = "The movie was great and I really enjoyed the performances of the actors."
sentiment = predict_sentiment(test_text, model, tokenizer, device)
print("The movie was great and I really enjoyed the performances of the actors.")
print(f"Predicted sentiment: {sentiment}")